In [7]:
import os
import pandas as pd
from catboost import CatBoostClassifier
import mlflow
from dotenv import load_dotenv
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score,
    recall_score, confusion_matrix, log_loss
)
from sklearn.model_selection import train_test_split


load_dotenv("/home/mle-user/mle_projects/mle-mlflow/.env")


EXPERIMENT_NAME = "churn_prediction"
RUN_NAME = "baseline_plus"
REGISTRY_MODEL_NAME = "churn_model_georgioparin"


TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
os.environ['AWS_ACCESS_KEY_ID'] = os.getenv('S3_ACCESS_KEY')
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("S3_SECRET_KEY")

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

df = pd.read_csv('~/mle_projects/mle-dvc/data/initial_data.csv')

y = df['target']
X = df.drop(columns=['id', 'begin_date', 'end_date', 'target'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cat_cols = list(X.select_dtypes(include=['object']).columns)
model = CatBoostClassifier(iterations=150, cat_features=cat_cols, verbose=0)
model.fit(X_train, y_train)

prediction = model.predict(X_test)
probas     = model.predict_proba(X_test)[:, 1]


pip_requirements = "/home/mle-user/mle_projects/mle-mlflow/requirements.txt"
signature = mlflow.models.infer_signature(X_test, prediction)
input_example = X_test[:10]
metadata = {'model_type': 'monthly'}

metrics = {}

metrics["auc"]       = roc_auc_score(y_test, probas)
metrics["precision"] = precision_score(y_test, prediction)
metrics["recall"]    = recall_score(y_test, prediction)

experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    mlflow.log_metrics(metrics)

    model_info = mlflow.catboost.log_model(
        cb_model=model,
        artifact_path="models",
        pip_requirements=pip_requirements,
        signature=signature,
        input_example=input_example,
        metadata = metadata,
        registered_model_name=REGISTRY_MODEL_NAME,
        await_registration_for=60
    )

/home/mle-user/.local/lib/python3.10/site-packages/mlflow/models/signature.py:212: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  inputs = _infer_schema(model_input) if model_input is not None else None
/home/mle-user/.local/lib/python3.10/site-packages/_distutils_hack/__init__.py:15: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replac

In [ ]:
TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
os.environ['AWS_ACCESS_KEY_ID'] = os.getenv('S3_ACCESS_KEY')
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("S3_SECRET_KEY")

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

client = mlflow.MlflowClient() 

REGISTRY_MODEL_NAME = "churn_model_georgioparin"


models = client.search_model_versions(
    filter_string=f"name = '{REGISTRY_MODEL_NAME}'")


print(f"Model info:\n {models}")

model_name_1 = models[1].name
model_version_1 = models[1].version
model_stage_1 = models[1].stage

model_name_2 = models[2].name
model_version_2 = models[2].version
model_stage_2 = models[2].stage


print(f"Текущий stage модели 1: {model_stage_1}")
print(f"Текущий stage модели 2: {model_stage_2}")

client.transition_model_version_stage(model_name_1, model_version_2, "production")
client.transition_model_version_stage(model_name_2, model_version_1, "staging")

# переимнуйте модель в реестре
client.REGISTRY_MODEL_NAME = "churn_model_georgioparin"

client.rename_registered_model(
    name=REGISTRY_MODEL_NAME, 
    new_name=f"{REGISTRY_MODEL_NAME}_b2c"
)